In [1]:
using Revise
using QuantumClifford

## Basics

In [55]:
m = S"
    -IYI
    XXX"

- _Y_
+ XXX

In [56]:
m.phases

2-element Vector{UInt8}:
 0x02
 0x00

In [57]:
m.xzs .|> bitstring

2×2 Matrix{String}:
 "0000000000000000000000000000000000000000000000000000000000000010"  …  "0000000000000000000000000000000000000000000000000000000000000111"
 "0000000000000000000000000000000000000000000000000000000000000010"     "0000000000000000000000000000000000000000000000000000000000000000"

In [65]:
typeof(m)

Stabilizer{Vector{UInt8}, Matrix{UInt64}}

In [58]:
m1 = copy(m)
canonicalize!(m1)

+iXZX
- _Y_

In [59]:
u = random_clifford(3)

X__ ⟼ + X__
_X_ ⟼ - XZZ
__X ⟼ + _Z_
Z__ ⟼ + ZZY
_Z_ ⟼ - X_X
__Z ⟼ + _XX


In [60]:
u * m

+ _ZY
- __Z

In [61]:
u * m1

-i_ZX
+ _ZY

In [54]:
canonicalize!(u * m1)

- X_Y
- ZZ_

## Clipping algorithm

### Goal: clipped gauge

1. $\rho_{\mathtt l}(x) + \rho_{\mathtt r}(x) = 2, \forall x$.
2. For all $x$ s.t. $\rho_{\mathtt l}(x) = 2$ or $\rho_{\mathtt r}(x) = 2$, the two stabilizers should be different.

### Steps

Step 1: pregauge.

1. $\rho_{\mathtt l}(x) \le 2, \forall x$.
2. For all $x$ s.t. $\rho_{\mathtt l}(x) = 2$, the two stabilizers should be different.

Setp 2: guage.

Do same thing to the right ends.

#### Algorithm

A variant of Gaussian elimination.

For the 1st qubit, try to find $g_i$ and $g_j$ nontrivially acting on it and $g_i \neq g_j$.

* If we can find $g_i$ and $g_j$, then use them to eliminate all other stabilizers on the 1st qubit. Note that we alway eliminate the longer stabliziers by shorter ones to make the other end not extended.
* If we cannot find such two stabilizers, then two possible cases:
    * Only one stabilizer acts nontrivially on the 1st qubit, then just pass.
    * More than one stabilizers act nontrivially on the 1st qubit, but all of them share a same Pauli operator on it.  So we can use eliminate all expect one.
    
And repeat the process to other qubits in the way of Gaussian elimination.